In [ ]:
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "scripts").is_dir() else Path.cwd().parent
DATA = ROOT / "data"


In [1]:
import pandas as pd

df = pd.read_csv(DATA / "processed" / "merged.csv", dtype={"session_id": "string"})

session_ids = df["session_id"].dropna().str.strip()
session_ids = session_ids[session_ids != ""]

print(f"Unique session_id values: {session_ids.nunique()}")

duplicate_counts = session_ids.value_counts()
duplicate_counts = duplicate_counts[duplicate_counts > 1]

print(f"Distinct IDs appearing more than once: {len(duplicate_counts)}")
display(duplicate_counts.rename("count").to_frame())

Unique session_id values: 701
Distinct IDs appearing more than once: 68


,count
session_id,
205469,80
205602,22
205230,16
205482,11
204562,10
...,...
205480,2
205519,2
204564,2


In [2]:
import pandas as pd

df = pd.read_csv(DATA / "processed" / "merged.csv", dtype={"session_id": "string"})
blank_session_ids = df["session_id"].isna() | df["session_id"].str.strip().eq("")
print(f"Blank or missing session_id values: {blank_session_ids.sum()}")

Blank or missing session_id values: 0


In [3]:
import pandas as pd

# Load the merged log and parse timestamps for chronological sorting.
df = pd.read_csv(DATA / "processed" / "merged.csv", dtype={"session_id": "string"})
df["Time"] = pd.to_datetime(df["Time"], errors="coerce")

# Exclude missing/blank IDs, then keep the earliest row for each session_id.
valid_sessions = df[df["session_id"].notna() & df["session_id"].str.strip().ne("")].copy()
unique_sessions = (
    valid_sessions.sort_values("Time")
    .drop_duplicates(subset="session_id", keep="first")
    .sort_values("Time")
    .reset_index(drop=True)
)

print(f"Rows in unique_sessions: {len(unique_sessions)}")
display(unique_sessions.head())

Rows in unique_sessions: 701


,labels,Time,Line,tsNs,labelTypes,id,session_id
0,"{""app"":""one"",""component"":""backend"",""container""...",2026-10-20 04:50:58,"Yeastar RAW body: ""{\""type\"":30031,\""sn\"":\""36...",1792446658589659600,"{""app"":""I"",""component"":""I"",""container"":""I"",""de...",1792446658589659600_32aa9158,205138
1,"{""app"":""one"",""component"":""backend"",""container""...",2026-10-20 05:05:51,"Yeastar RAW body: ""{\""type\"":30031,\""sn\"":\""36...",1792447551825294800,"{""app"":""I"",""component"":""I"",""container"":""I"",""de...",1792447551825294800_f52d6686,205139
2,"{""app"":""one"",""component"":""backend"",""container""...",2026-10-20 05:48:41,"Yeastar RAW body: ""{\""type\"":30031,\""sn\"":\""36...",1792450121306766100,"{""app"":""I"",""component"":""I"",""container"":""I"",""de...",1792450121306766100_83dcee5,205140
3,"{""app"":""one"",""component"":""backend"",""container""...",2026-10-20 06:06:08,"Yeastar RAW body: ""{\""type\"":30031,\""sn\"":\""36...",1792451168616017300,"{""app"":""I"",""component"":""I"",""container"":""I"",""de...",1792451168616017300_7cf2d16,205141
4,"{""app"":""one"",""component"":""backend"",""container""...",2026-10-20 06:08:00,"Yeastar RAW body: ""{\""type\"":30031,\""sn\"":\""36...",1792451280794326200,"{""app"":""I"",""component"":""I"",""container"":""I"",""de...",1792451280794326200_7743a027,205142


In [4]:
df_yeastar = pd.read_csv(DATA / "reference" / "YeastarLog20Okt.csv")
df_yeastar.head()

,session_id,first_msg_time,last_msg_time
0,205138,20/10/2026 04:50:58,20/10/2026 12:58:50
1,205139,20/10/2026 05:05:51,20/10/2026 08:51:51
2,205140,20/10/2026 05:48:41,20/10/2026 11:19:22
3,205141,20/10/2026 06:06:08,20/10/2026 06:07:10
4,205142,20/10/2026 06:08:00,20/10/2026 06:08:54


In [ ]:
import pandas as pd

# Load both datasets with IDs as strings so the values compare consistently.
df = pd.read_csv(DATA / "processed" / "merged.csv", dtype={"session_id": "string"})
df_yeastar = pd.read_csv(DATA / "reference" / "YeastarLog20Okt.csv", dtype={"session_id": "string"})

df_ids = set(df["session_id"].dropna().str.strip())
df_ids.discard("")
yeastar = df_yeastar.copy()
yeastar["session_id"] = yeastar["session_id"].str.strip()
yeastar = yeastar[yeastar["session_id"].notna() & yeastar["session_id"].ne("")]

# Yeastar IDs not present in df, preserving their first and last message times.
unmatched_sessions = (
    yeastar[~yeastar["session_id"].isin(df_ids)]
    .drop_duplicates(subset="session_id")
    .loc[:, ["session_id", "first_msg_time", "last_msg_time"]]
    .sort_values("session_id")
    .reset_index(drop=True)
)

display(unmatched_sessions.head())

Unmatched Yeastar session IDs: 30
Saved to unmatched_yeastar_sessions.csv


,session_id,first_msg_time,last_msg_time
0,204421,20/10/2026 09:29:09,20/10/2026 09:29:09
1,204554,20/10/2026 09:15:00,20/10/2026 09:15:00
2,204560,20/10/2026 06:18:43,20/10/2026 08:52:38
3,204563,20/10/2026 09:02:35,20/10/2026 09:02:35
4,204569,20/10/2026 08:18:58,20/10/2026 08:57:26


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 979 entries, 0 to 978
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   labels      979 non-null    object
 1   Time        979 non-null    object
 2   Line        979 non-null    object
 3   tsNs        979 non-null    int64 
 4   labelTypes  979 non-null    object
 5   id          979 non-null    object
 6   session_id  979 non-null    string
dtypes: int64(1), object(5), string(1)
memory usage: 53.7+ KB


In [7]:
df_yeastar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 673 entries, 0 to 672
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   session_id      673 non-null    string
 1   first_msg_time  673 non-null    object
 2   last_msg_time   673 non-null    object
dtypes: object(2), string(1)
memory usage: 15.9+ KB


In [8]:
import pandas as pd

df = pd.read_csv(DATA / "reference" / "unmatched_yeastar_sessions.csv", dtype={"session_id": "string"})
df["first_msg_time"] = pd.to_datetime(df["first_msg_time"], errors="coerce")

on_september_23 = df[df["first_msg_time"].dt.strftime("%m-%d").eq("10-20")] # edit
session_ids = on_september_23["session_id"].dropna().str.strip()
session_ids = session_ids[session_ids.ne("")]

print(f"Unmatched sessions with first_msg_time: {len(on_september_23)}")
print(f"Session IDs ({len(session_ids)} rows):")
for session_id in session_ids:
    print(session_id)

Unmatched sessions with first_msg_time: 30
Session IDs (30 rows):
204421
204554
204560
204563
204569
204637
204710
204745
204750
204782
204821
204830
204892
204985
204991
204999
205001
205003
205011
205035
205037
205039
205040
205055
205109
205110
205111
205193
205261
205405


C:\Users\celin\AppData\Local\Temp\ipykernel_22164\2916086417.py:4: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["first_msg_time"] = pd.to_datetime(df["first_msg_time"], errors="coerce")


In [16]:
df.info

<bound method DataFrame.info of     session_id      first_msg_time        last_msg_time
0       191249 2026-09-23 17:15:00  2026-09-23 17:15:00
1       191395 2026-09-23 10:15:00  2026-09-23 10:15:00
2       191955 2026-09-23 15:15:00  2026-09-23 15:15:00
3       191957 2026-09-23 09:15:00  2026-09-23 09:15:00
4       191988 2026-09-23 09:15:00  2026-09-23 09:15:00
..         ...                 ...                  ...
215     203134 2026-10-15 11:09:04  2026-10-15 11:09:11
216     203135 2026-10-15 11:09:41  2026-10-15 11:10:27
217     203136 2026-10-15 11:09:46  2026-10-15 11:11:02
218     203137 2026-10-15 11:11:20  2026-10-15 11:13:36
219     203138 2026-10-15 11:12:14  2026-10-15 11:13:31

[220 rows x 3 columns]>